# Fase 2 — Modelagem Estocástica (SimPy)
### Validação cruzada do R-UDP Go-Back-N

Anthony Irlan Marques Luz · PPGCC/UFPI · Projeto de Redes 2026-1

Este notebook integra as tarefas de validação **D3–D9** do simulador de eventos discretos em SimPy, todas ancoradas nos dados reais da Fase 1 (`results/tables/summary_stats.csv`). Cada seção corresponde a uma tarefa do §3.1 da descrição e gera seu gráfico em Plotly.

| Seção | Tarefa §3.1 | Figura |
|---|---|---|
| D3 Calibração | 1,2,3,6 | tempo e retx sim×real |
| D4 Convergência | 10 | IC 95% bootstrap |
| D5 Vazão | 4 | vazão × tamanho |
| D6 Janela | 5 | vazão × N |
| D7 Jitter | 7 | variância × jitter |
| D8 Estresse | 8 | previsão 25% perda |
| D9 Eficiência | 9 | razão DATA/ACK |

> **Como usar:** *Runtime → Run all*. A primeira célula faz o bootstrap (clona o repositório e instala dependências) quando executada no Google Colab; localmente ela apenas ajusta o `sys.path`.

## 0. Setup (Colab + dependências)

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Anth0nYM/redes-de-computadores-dccmapi.git"
REPO_DIR = "redes-de-computadores-dccmapi"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )

# Garante a raiz do repositório no path (suporta rodar de notebooks/).
if not os.path.isdir("src") and os.path.isdir(os.path.join("..", "src")):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import plotly.io as pio
if IN_COLAB:
    pio.renderers.default = "colab"

print("cwd:", os.getcwd(), "| Colab:", IN_COLAB)

## D3 — Calibração (tarefas 1, 2, 3, 6)
O simulador reproduz tempo e retransmissões reais sem fatores de ajuste: o atraso é Normal(μ, jitter), a perda é Bernoulli com a perda *efetiva* `1-(1-p)³` (3 fragmentos IP por datagrama de 4112 B).

In [ ]:
from src.sim.validate import CalibrationValidator, build_time_figure, build_retx_figure

print(CalibrationValidator(num_reps=30).generate_report())
build_time_figure(num_reps=30).show()
build_retx_figure(num_reps=30).show()

## D4 — Convergência estatística (tarefa 10)
Com ≥30 repetições estimamos um intervalo de confiança de 95% (bootstrap percentil) sobre o tempo médio. O critério de aceite é o IC cobrir a medição real.

In [ ]:
from src.sim.converge import ConvergenceAnalyzer, build_figure as converge_fig

print(ConvergenceAnalyzer(num_reps=30, seed=1).generate_report())
converge_fig(num_reps=30, seed=1).show()

## D5 — Curva de vazão (tarefa 4)
A vazão de um protocolo limitado pela janela satura em `janela × bloco / RTT`. Arquivos maiores amortizam o custo fixo de setup e aproximam a vazão do teto teórico.

In [ ]:
from src.sim.throughput import build_figure as throughput_fig

throughput_fig(scenarios=["A", "B"], reps=3).show()

## D6 — Sensibilidade da janela N (tarefa 5)
Sob um gargalo de banda finito o tubo comporta `BDP = banda × RTT` bytes. Abaixo da janela `N* = BDP/bloco` o protocolo é limitado pela janela; acima, pelo enlace — daí o joelho na curva.

In [ ]:
from src.sim.window_sweep import build_figure as window_fig

window_fig(bandwidth_KBps=2000.0, reps=4).show()

## D7 — Impacto do jitter (tarefa 7)
Em canal limpo (cenário A, sem perda) variamos o jitter (desvio do atraso). Mais jitter espalha cada amostra de RTT, aumentando a variância do tempo total de transferência.

In [ ]:
from src.sim.jitter_sweep import JitterSweep, build_figure as jitter_fig

sweep = JitterSweep(seed=1)
jit, means, stds = sweep.compute()
for j, m, s in zip(jit, means, stds):
    print(f"jitter={j:3.0f} ms -> tempo {m:.3f} s, desvio {s:.4f} s")
print("variância cresce com o jitter?", sweep.variance_increases())
jitter_fig(seed=1).show()

## D8 — Cenário de estresse 25% (tarefa 8)
A perda do `tc` é por pacote (bruta); o simulador consome a efetiva `1-(1-p)³`. Mantendo o atraso de C (100 ms), previmos o tempo a 25% de perda bruta (p_ef ≈ 57,8%) — acima do cenário C, na tendência monotônica.

In [ ]:
from src.sim.stress import StressForecast, build_figure as stress_fig

print(StressForecast(reps=30, seed=1).generate_report())
stress_fig(reps=30, seed=1).show()

## D9 — Eficiência DATA/ACK (tarefa 9)
O receptor GBN emite um ACK cumulativo por pacote que **chega**, então a razão DATA/ACK segue `1/(1-p_ef)`: ~1 em canal limpo e crescente com a perda. A eficiência piora à medida que a perda aumenta.

In [ ]:
from src.sim.efficiency import EfficiencyStudy, build_figure as efficiency_fig

print(EfficiencyStudy(reps=20, seed=1).generate_report())
efficiency_fig(reps=20, seed=1).show()

## Conclusão
As sete seções fecham as 10 tarefas de validação do §3.1 (D1/D2 são o canal e o motor de eventos, exercitados pelos testes em `tests/`). O simulador reproduz os pontos reais dentro do IC 95% e expõe, por varredura de parâmetros, comportamentos não medidos no enlace real (saturação de janela, impacto do jitter, estresse de perda e eficiência do controle).